# How to Think About Group Operations

> **Big idea — split-apply-combine**: the standard pattern behind every group operation.
> 1. **Split** — break a pandas object (Series, DataFrame, ...) into groups based on one or more *keys*.
> 2. **Apply** — run a function on each group independently (e.g. `mean`, `sum`, or a custom function).
> 3. **Combine** — stitch the per-group results back into a single result object.
>
> The shape of the final result depends entirely on what happens in the "apply" step — sometimes you get back a scalar per group, sometimes a whole DataFrame per group.

*Insert Illustration of a group aggregation here*

## What can a "key" be?

Each grouping key can take many forms, and the keys passed to a single `groupby` call don't even have to all be the same type:

| Key type | What it looks like | Example |
|---|---|---|
| Column name (or list of names) | A string naming column(s) in the DataFrame | `df.groupby("key1")` |
| Array/list of values | Same length as the axis being grouped | `df.groupby(states_array)` |
| Dict or Series | Maps existing axis labels → new group names | `people.groupby({"a": "red", "b": "blue"})` |
| Function | Called once per axis label; return value becomes the group name | `people.groupby(len)` |

The last three are really shortcuts: under the hood, pandas converts each of them into an array of group labels before doing the actual splitting. Don't worry if this feels abstract yet — the examples below make it concrete.

To start, here's a small tabular dataset as a DataFrame:

In [1]:
import pandas as pd 
import numpy as np 


In [2]:
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                "data1": np.random.standard_normal(7),
                "data2": np.random.standard_normal(7)})


df

,key1,key2,data1,data2
0,a,1,-0.657094,0.053192
1,a,2,-1.249411,0.826502
2,NaN,1,0.846296,-2.440322
3,b,2,0.343711,-0.928842
4,b,1,0.827680,-0.009234
5,a,<NA>,0.309267,-1.253450
6,NaN,1,-0.626442,-0.403889


**A note on the missing data here:** `key1` uses Python's `None` (which shows up as `NaN` in an object/string column), while `key2` uses pandas' nullable `Int64` dtype, where missing values display as `<NA>`. Both count as "missing" to `groupby` — and by default, **rows whose group key is missing get dropped from the result entirely** (more on this, and how to opt out with `dropna=False`, further down).

## GroupBy Basics: Computing a Group Mean

Suppose you wanted to compute the mean of the `data1` column using the labels from `key1`. There are a number of ways to do this. One is to access `data1` and call `groupby` with the column at `key1`:

In [3]:
grouped = df["data1"].groupby(df["key1"])

grouped

This `grouped` variable is now a special "GroupBy" object. It has not actually computed anything yet except for some intermediate data about the group key `df["key1"]`. The idea is that the object has all of the information needed to then apply some operation to each of the groups. For example, to compute group means we can call the GroupBy's `mean` method:

In [4]:
grouped.mean()

key1
a   -0.532413
b    0.585696
Name: data1, dtype: float64

`mean()` is just one of many built-in aggregation methods a GroupBy object supports:

| Method | What it computes |
|---|---|
| `count()` | number of non-null values per group |
| `size()` | number of rows per group (NaNs included) |
| `sum()` | sum of values |
| `mean()` / `median()` | central tendency |
| `std()` / `var()` | spread |
| `min()` / `max()` | extremes |
| `describe()` | all of the above at once, as a summary table |

Any of these can be substituted for `.mean()` in the examples below.

If instead we had passed multiple arrays as a list, we'd get something different:

In [5]:
means=df["data1"].groupby([df["key1"], df["key2"]]).mean()

means

key1  key2
a     1      -0.657094
      2      -1.249411
b     1       0.827680
      2       0.343711
Name: data1, dtype: float64

In [6]:
means.unstack()

key2,1,2
key1,,
a,-0.657094,-1.249411
b,0.827680,0.343711


`unstack()` pivots the innermost level of a hierarchical (multi-level) index into columns, turning the "long" `Series` above into a "wide" `DataFrame` — often easier to read. Its inverse is `stack()`.

In this example, the group keys are all Series, though they could be any arrays of the right length:

In [7]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])

years = [2005, 2005, 2006, 2205, 2006, 2005, 2006]

df["data1"].groupby([states, years]).mean()

CA  2005   -0.470072
    2006    0.846296
OH  2005   -0.657094
    2006    0.100619
    2205    0.343711
Name: data1, dtype: float64

## Aggregating an Entire DataFrame

So far we've grouped a single `Series` (`df["data1"]`). You can just as easily call `groupby` directly on the whole `DataFrame` — the aggregation is then applied to every remaining column at once.

In [8]:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.532413,-0.124585
b,1.5,0.585696,-0.469038


In [9]:
df.groupby("key2").mean(numeric_only=True)

,data1,data2
key2,,
1,0.09761,-0.700063
2,-0.45285,-0.051170


**Why does grouping by `key2` need `numeric_only=True` but grouping by `key1` doesn't?** When you group by `"key1"`, the `key1` column becomes the *index* of the result and is no longer one of the columns being averaged — the remaining columns (`key2`, `data1`, `data2`) are all numeric, so `.mean()` just works. But when you group by `"key2"`, the leftover columns are `key1` (strings), `data1`, and `data2` — and pandas won't silently average a text column. As of pandas 2.0+, `numeric_only` defaults to `False` for most aggregations (older pandas used to drop non-numeric columns for you automatically), so you now have to opt in explicitly whenever a non-numeric column could get swept into the aggregation.

## Group Sizes and Counts

In [10]:
df.groupby(["key1", "key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

Note that any missing values in a group key are excluded from the result by default. This behaviour can be disabled by passing `dropna=False` to `groupby`:

In [11]:
df.groupby("key1", dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [12]:
df.groupby(["key1", "key2"], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

A group function similar in spirit to `size` is count, which computes the number of nonnull values in each group:

In [13]:
df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


## Iterating over Groups

The object returned by `groupby` supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data. Consider the following:

In [14]:
for name, group in df.groupby("key1"):
    print(name)
    print(group)

a
  key1  key2     data1     data2
0    a     1 -0.657094  0.053192
1    a     2 -1.249411  0.826502
5    a  <NA>  0.309267 -1.253450
b
  key1  key2     data1     data2
3    b     2  0.343711 -0.928842
4    b     1  0.827680 -0.009234


In the case of multiple keys, the first element in the tuple will be a tuple of key values:

In [15]:
for (k1, k2), group in df.groupby(["key1", "key2"]):
    print((k1, k2))
    print(group)

('a', 1)
  key1  key2     data1     data2
0    a     1 -0.657094  0.053192
('a', 2)
  key1  key2     data1     data2
1    a     2 -1.249411  0.826502
('b', 1)
  key1  key2    data1     data2
4    b     1  0.82768 -0.009234
('b', 2)
  key1  key2     data1     data2
3    b     2  0.343711 -0.928842


Of course, you can choose to do whatever you want with the pieces of data. A recipe you may find useful is computing a dictionary of the data pieces as a one-liner:

In [16]:
pieces = {name : group for name, group in df.groupby("key1")}

pieces["b"]

,key1,key2,data1,data2
3,b,2,0.343711,-0.928842
4,b,1,0.827680,-0.009234


## Grouping on the Column Axis

`groupby` normally splits along the row axis, but you can just as easily group the *columns* instead. For example, we could group the columns of our example `df` by whether they start with `"key"` or `"data"`.

(In older pandas you'd pass `axis="columns"` directly to `groupby`; that argument was removed in pandas 2.0+. The modern equivalent — used below — is to transpose the DataFrame first, group on the now-row axis, then transpose back if you need the original orientation.)

In [17]:
grouped = df.T.groupby({"key1": "key", "key2": "key",
                    "data1": "data", "data2": "data"})



We can print out the groups like so:

In [18]:
for group_key, group_values in grouped:
    print(group_key)
    print(group_values)

data
              0         1         2         3         4         5         6
data1 -0.657094 -1.249411  0.846296  0.343711   0.82768  0.309267 -0.626442
data2  0.053192  0.826502 -2.440322 -0.928842 -0.009234  -1.25345 -0.403889
key
      0  1    2  3  4     5    6
key1  a  a  NaN  b  b     a  NaN
key2  1  2    1  2  1  <NA>    1


## Selecting a Column or Subset of Columns

Indexing a GroupBy object created from a DataFrame with a column name or array of column names has the effect of column subsetting for aggregation. This means that:

In [19]:
df.groupby("key1")["data1"]
df.groupby("key1")["data2"]

(Note: in a Jupyter cell, only the *last* expression auto-prints — both lines above are valid and equivalent in form, but only the `"data2"` result actually displays.)

are conveniences for:

In [20]:
df["data1"].groupby(df["key1"])
df[["data2"]].groupby(df["key2"])

**Why bother subsetting columns before aggregating?** Especially for large datasets, it may be desirable to aggregate only a few columns. For example, to compute just the means for the `data2` column and get the result back as a DataFrame:

In [21]:
df.groupby(["key1", "key2"])[["data2"]].mean()

data2
key1 key2          
a    1     0.053192
     2     0.826502
b    1    -0.009234
     2    -0.928842

In [22]:
s_grouped = df.groupby(["key1", "key2"])["data2"]

s_grouped

In [23]:
s_grouped.mean()

key1  key2
a     1       0.053192
      2       0.826502
b     1      -0.009234
      2      -0.928842
Name: data2, dtype: float64

## Grouping with Dictionaries and Series

Grouping information may exist in a form other than an array. Let's consider another example DataFrame:

In [24]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                    columns=["a", "b", "c", "d", "e"],
                    index=["Joe", "Steve", "Wanda", "Jill", "Trey"])

people.iloc[2:3, [1, 2]] = np.nan 

people

,a,b,c,d,e
Joe,0.511152,-1.398299,-1.311804,-0.237421,1.538211
Steve,0.120361,0.912907,0.717848,0.683176,0.603476
Wanda,1.217833,NaN,NaN,0.182056,0.853214
Jill,0.836849,-0.571239,-1.978889,-1.215905,0.336054
Trey,0.285759,1.795751,-0.545531,0.749140,-0.081804


In [25]:
mapping = {"a": "red", "b": "red", "c": "blue",
        "d" : "blue", "e": "red", "f": "orange"}

Now you could construct an array from this dictionary to pass to `groupby`, but instead we can just pass the dictionary directly (again using the transpose pattern, since we're grouping the *columns* of `people` here):

In [26]:
by_column = people.T.groupby(mapping)

by_column.sum()

,Joe,Steve,Wanda,Jill,Trey
blue,-1.549225,1.401025,0.182056,-3.194794,0.203609
red,0.651065,1.636745,2.071048,0.601664,1.999706


**Gotcha:** `mapping` has an extra key, `"f"`, that doesn't correspond to any column in `people` — that's fine, unused keys are just ignored. The reverse situation is *not* fine: if `people` had a column with no entry in `mapping`, `groupby` would raise a `KeyError`. A `Series` works the same way as a dict here and is convenient when pandas needs to check that the index is being used consistently.

## Grouping with Functions

Using Python functions is a more generic way of defining a group mapping compared with a dictionary or Series. Any function passed as a group key will be called once per index label, with the return value used as that label's group name.

Suppose you wanted to group by name length. While you could compute an array of string lengths yourself, it's simpler to just pass the `len` function:

In [27]:
people.groupby(len).sum()

,a,b,c,d,e
3,0.511152,-1.398299,-1.311804,-0.237421,1.538211
4,1.122608,1.224511,-2.524420,-0.466765,0.254250
5,1.338195,0.912907,0.717848,0.865232,1.456690


Mixing functions with arrays, dictionaries, or Series is not a problem, as everything gets converted to array internally. 

In [28]:
key_list = ["one", "one", "one", "two", "two"]

people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,0.511152,-1.398299,-1.311804,-0.237421,1.538211
4,two,0.285759,-0.571239,-1.978889,-1.215905,-0.081804
5,one,0.120361,0.912907,0.717848,0.182056,0.603476


A `lambda` works just as well as a named function, which is handy for one-off grouping logic that isn't worth writing a whole function for. For example, grouping by the number of vowels in each name:

In [29]:
vowels = "aeiou"

people.groupby(lambda name: sum(1 for ch in name.lower() if ch in vowels)).mean()

,a,b,c,d,e
1,0.561304,0.612256,-1.262210,-0.233383,0.127125
2,0.616449,-0.242696,-0.296978,0.209270,0.998301


## Grouping by Index Levels

A **MultiIndex** (hierarchical index) lets a single axis carry more than one layer of labels — e.g. columns labeled by both `cty` (country) and `tenor` (a bond's time-to-maturity), as below. A final convenience for hierarchically indexed datasets is the ability to aggregate using one of those index *levels* directly, without first flattening the index yourself. Let's look at an example:

In [30]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                    [1, 2, 5, 1, 3]],
                                    names=["cty", "tenor" ])

hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)

In [31]:
hier_df

cty          US                            JP          
tenor         1         2         5         1         3
0      1.957505  0.510871  1.480006 -1.162530  1.702838
1      0.069476  0.576342 -0.859528  0.022000  0.842507
2     -1.435987  0.173365 -0.430235 -0.017502 -0.535146
3      1.366497 -1.505163  0.579710 -0.081300 -0.399859

To pass by level, pass the level number or name using the `level` keyword:

In [32]:
hier_df.T.groupby(level="cty").count().T

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


Note the `.T` on both ends: `axis="columns"` was removed from `groupby()` in pandas 2.0+, so the modern pattern for grouping *columns* instead of rows is always **transpose → group (rows) → transpose back**. You'll see this same `.T` trick used earlier in the [Grouping with Dictionaries and Series](#Grouping-with-Dictionaries-and-Series) section.

---

## Summary / Cheat Sheet

**What can you group by?**

| Key type | Example |
|---|---|
| Column name(s) | `df.groupby("key1")` / `df.groupby(["key1", "key2"])` |
| Array/list matching axis length | `df.groupby([states, years])` |
| Dict or Series (old labels → new group names) | `people.T.groupby(mapping)` |
| Function (called once per label) | `people.groupby(len)` |
| Index level (for MultiIndex) | `hier_df.T.groupby(level="cty")` |

**Common gotchas hit in this notebook:**
- `numeric_only=True` is required whenever a non-numeric column could get swept into `.mean()`/`.sum()`/etc. — pandas no longer drops it for you silently.
- `axis="columns"` no longer exists on `groupby()`. Transpose first, group on rows, transpose back if needed.
- Rows with a **missing group key** are dropped by default — pass `dropna=False` to keep them (shown up under `NaN`/`<NA>` group labels).
- `size()` counts rows per group; `count()` counts non-null values per *column*, per group — they can disagree when there's missing data.
- `df.groupby(...)["col"]` and `df["col"].groupby(...)` are equivalent — the bracket form is just a convenience for subsetting columns before aggregating (handy for performance on wide DataFrames).
